# Datamart Month Acquiring — FinalDF + Excel Compare (декомпозиция)

Тетрадка разбита на независимые этапы, чтобы при ошибке перезапускать только проблемную секцию, а не весь pipeline.

In [ ]:
import re
import time
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return s if s else None


def to_decimal_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def exact_match_rate_from_abs(abs_series, tol=0.0):
    if len(abs_series) == 0:
        return 0.0
    return round((abs_series.fillna(0) <= tol).mean() * 100, 2)

In [ ]:
# Параметры и подключение
report_month = '2026-04-01'
excel_path = '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx'
excel_header = 0

run_invalidate_metadata = False
run_excel_compare = True
show_compare_top = True
save_final_csv = False
output_csv_path = './datamart_month_acquiring_final_2026_04.csv'

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')
snapshot_month_start = month_start

print(f'report_month={report_month}, month_start={month_start}, month_end={month_end}')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

invalidate_tables = [
    'ods_alpha.scd1_agreements', 'ods_alpha.scd1_companies', 'ods_alpha.scd1_agr_terms',
    'ocrm_ul.s_org_ext', 'cdiul.ext_id_org', 'ods_alpha.scd1_merchants',
    'ods_alpha.scd1_pos_terminals', 'sandbox_ai.shestopalov_terminal_amortization_oneoff',
    'ods_alpha.scd1_trx', 'ods_alpha.scd1_trx_acq', 'ods_alpha.scd1_trx_int',
    'ods_alpha.scd1_base24_fiids', 'ods.scd1_z_r2_ip_merchants', 'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix', 'ods.scd1_z_cl_corp', 'ods.scd1_z_depart',
    'ods.scd1_z_branch', 'ods.scd1_z_r2_tariff_plan'
]

if run_invalidate_metadata:
    with imp:
        for t in invalidate_tables:
            imp.execute(f'invalidate metadata {t}')
    print('Invalidate metadata completed')
else:
    print('Invalidate metadata skipped')

## Секция 1. Формирование `final_df` (по шагам)

Ниже каждый этап вынесен в отдельную ячейку:
`01_sa_perimeter` -> `02_cdi_map` -> `03_cft_map` -> `04_operational_metrics` -> `05_transaction_metrics` -> `06_r2_legacy_attrs` -> `07_base_merge` -> `08_agr_fallback` -> `08b_tariff_fix_map` -> `09_actual_tariff_by_agr` -> `10_apply_tariff_fix_and_formulas`.

In [ ]:
# 01_sa_perimeter
sql_sa_perimeter = f"""
select distinct
  cast(a.n_agr as string) as n_agr,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
  and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
  )
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    sa_df = imp.fetch(sql_sa_perimeter)

if sa_df is None:
    sa_df = pd.DataFrame()
if not sa_df.empty:
    sa_df['inn'] = sa_df['inn'].map(normalize_inn)
    sa_df['contract_number'] = sa_df['contract_number'].map(normalize_contract)

print(f'SA perimeter rows: {len(sa_df):,}')
display(sa_df.head(3))

In [ ]:
# 02_cdi_map
if 'sa_df' not in globals():
    raise RuntimeError('Сначала выполните 01_sa_perimeter')

inn_values = sorted([x for x in sa_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
inn_sql_list = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_cdi = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    trim(cast(soe.x_area_resp as string)) as x_area_resp_norm,
    case
      when soe.x_area_resp like 'ДММБ%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ(ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ (ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДСБ%' then 'ДМСБ'
      when soe.x_area_resp like 'ДКБ%' then 'ДКБ'
      when soe.x_area_resp like 'ДРПА%' then 'ДРПА'
      when soe.x_area_resp like 'ДРА%' then 'ДРПА'
      when lower(soe.x_area_resp) like 'не под%' then 'Не подлежит сегментации'
      when nullif(trim(cast(soe.x_area_resp as string)), '') is null then null
      else null
    end as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc, cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm, x_area_resp_norm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  o.x_area_resp_norm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cdi_map_df = imp.fetch(sql_cdi)

if cdi_map_df is None:
    cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])
if not cdi_map_df.empty:
    cdi_map_df['inn'] = cdi_map_df['inn'].map(normalize_inn)
    cdi_map_df['cdi_id'] = cdi_map_df['cdi_id'].astype(str)
    allowed = {'ДМ', 'ДМСБ', 'ДКБ', 'ДРПА', 'Не подлежит сегментации'}
    cdi_map_df['ssp_ocrm'] = cdi_map_df['ssp_ocrm'].where(cdi_map_df['ssp_ocrm'].isin(allowed), None)
    cdi_map_df = cdi_map_df.drop_duplicates(subset=['inn'], keep='first')

print(f'CDI rows: {len(cdi_map_df):,}')
display(cdi_map_df.head(3))

In [ ]:
# 03_cft_map
if 'cdi_map_df' not in globals():
    raise RuntimeError('Сначала выполните 02_cdi_map')

cdi_values = sorted([x for x in cdi_map_df.get('cdi_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cdi_sql_list = ', '.join([f"'{x}'" for x in cdi_values]) if cdi_values else "''"

sql_cft = f"""
select
  cast(e.party_id as string) as cdi_id,
  cast(e.cmo_ext_party_source_id as string) as cft_id
from cdiul.ext_id_org e
where cast(e.party_id as string) in ({cdi_sql_list})
  and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cft_map_df = imp.fetch(sql_cft)

if cft_map_df is None:
    cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])
if not cft_map_df.empty:
    cft_map_df['cdi_id'] = cft_map_df['cdi_id'].astype(str)
    cft_map_df['cft_id'] = cft_map_df['cft_id'].astype(str)
    cft_map_df = cft_map_df.drop_duplicates(subset=['cdi_id'], keep='first')

print(f'CFT rows: {len(cft_map_df):,}')
display(cft_map_df.head(3))

In [ ]:
# 04_operational_metrics
if 'sa_df' not in globals():
    raise RuntimeError('Сначала выполните 01_sa_perimeter')

if sa_df.empty:
    cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization'])
else:
    n_agrs = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
    agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

    sql_cmp = f"""
    with sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr, cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({agr_in})
    ),
    m as (
      select sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants mm
        on cast(mm.n_cmp as string) = sa.n_cmp_client
      where mm.c_nmrc is not null and coalesce(mm.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string)
    ),
    term_active as (
      select m.n_agr, m.n_cmp_client, m.c_nmrc, cast(t.c_nter as string) as c_nter
      from m
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = m.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
      group by m.n_agr, m.n_cmp_client, m.c_nmrc, cast(t.c_nter as string)
    ),
    retl as (
      select n_agr, count(distinct c_nmrc) as retl_cnt
      from term_active
      group by n_agr
    ),
    term as (
      select n_agr, count(distinct c_nter) as term_cnt
      from term_active
      group by n_agr
    ),
    amort as (
      select ta.n_agr, sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization
      from term_active ta
      left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
        on cast(am.c_nter as string) = ta.c_nter
      group by ta.n_agr
    )
    select sa.n_agr, sa.n_cmp_client, r.retl_cnt, t.term_cnt, a.amortization
    from sa_agr sa
    left join retl r on r.n_agr = sa.n_agr
    left join term t on t.n_agr = sa.n_agr
    left join amort a on a.n_agr = sa.n_agr
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        cmp_df = imp.fetch(sql_cmp)

if cmp_df is None:
    cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization'])
if not cmp_df.empty:
    cmp_df['n_agr'] = cmp_df['n_agr'].astype(str)
    cmp_df['n_cmp_client'] = cmp_df['n_cmp_client'].astype(str)

print(f'Operational rows: {len(cmp_df):,}')
display(cmp_df.head(3))

In [ ]:
# 05_transaction_metrics
if 'sa_df' not in globals():
    raise RuntimeError('Сначала выполните 01_sa_perimeter')

if sa_df.empty:
    trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt'])
else:
    n_agrs = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
    agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

    sql_trx = f"""
    with sa_agr as (
      select distinct cast(n_agr as string) as n_agr, cast(n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements
      where cast(n_agr as string) in ({agr_in})
    ),
    trx_base_raw as (
      select cast(t.n_trx as string) as n_trx, cast(t.c_nter as string) as c_nter, cast(t.n_amt_src as double) as n_amt_src
      from ods_alpha.scd1_trx t
      left join ods_alpha.scd1_base24_fiids fa
        on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
      where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
        and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx_base as (
      select n_trx, max(c_nter) as c_nter, max(n_amt_src) as n_amt_src
      from trx_base_raw
      group by n_trx
    ),
    ta_raw as (
      select cast(a.n_trx as string) as n_trx, cast(a.n_agr as string) as n_agr, coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
      from ods_alpha.scd1_trx_acq a
      join trx_base tb on tb.n_trx = cast(a.n_trx as string)
      where cast(a.n_agr as string) in ({agr_in})
    ),
    ta as (
      select n_trx, n_agr, max(n_amt_tax) as n_amt_tax
      from ta_raw
      group by n_trx, n_agr
    ),
    trx_keys as (
      select distinct n_trx
      from ta
    ),
    trx_int_agg as (
      select cast(i.n_trx as string) as n_trx, sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join trx_keys k on k.n_trx = cast(i.n_trx as string)
      group by cast(i.n_trx as string)
    ),
    tj as (
      select ta.n_agr, sa.n_cmp_client, ta.n_trx, tb.c_nter, tb.n_amt_src, ta.n_amt_tax
      from ta
      join trx_base tb on tb.n_trx = ta.n_trx
      left join sa_agr sa on sa.n_agr = ta.n_agr
    ),
    trx_agg as (
      select
        tj.n_agr,
        count(distinct tj.n_trx) as trx_cnt,
        sum(tj.n_amt_src) as trx_sum,
        sum(tj.n_amt_tax) as commission_from_ops,
        sum(coalesce(i.n_amt_fee, 0.0)) as int_component
      from tj
      left join trx_int_agg i on i.n_trx = tj.n_trx
      group by tj.n_agr
    ),
    active_term_agg as (
      select
        tj.n_cmp_client,
        count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_term_cnt
      from tj
      where tj.n_cmp_client is not null
      group by tj.n_cmp_client
    )
    select
      t.n_agr,
      t.trx_cnt,
      t.trx_sum,
      t.commission_from_ops,
      t.int_component,
      a.n_cmp_client,
      a.active_term_cnt
    from trx_agg t
    left join sa_agr sa on sa.n_agr = t.n_agr
    left join active_term_agg a on a.n_cmp_client = sa.n_cmp_client
    """

    with imp:
        imp.execute('set MEM_LIMIT=16g')
        trx_df = imp.fetch(sql_trx)

if trx_df is None:
    trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt'])
if not trx_df.empty:
    trx_df['n_agr'] = trx_df['n_agr'].astype(str)
    trx_df['n_cmp_client'] = trx_df['n_cmp_client'].astype(str)

active_term_df = (
    trx_df[['n_cmp_client', 'active_term_cnt']]
    .dropna(subset=['n_cmp_client'])
    .drop_duplicates(subset=['n_cmp_client'])
    if len(trx_df)
    else pd.DataFrame(columns=['n_cmp_client', 'active_term_cnt'])
)

print(f'Transaction rows: {len(trx_df):,}')
display(trx_df.head(3))

In [ ]:
# 06_r2_legacy_attrs
if 'cft_map_df' not in globals():
    raise RuntimeError('Сначала выполните 03_cft_map')

cft_values = sorted([x for x in cft_map_df.get('cft_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cft_sql_list = ', '.join([f"'{x}'" for x in cft_values]) if cft_values else "''"

if not cft_values:
    r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])
else:
    sql_r2 = f"""
    with r2 as (
      select
        cast(m.id as string) as r2_id,
        cast(m.c_cl_org as string) as cft_id,
        cast(m.c_depart as string) as c_depart,
        cast(m.c_tariff_plan as string) as c_tariff_plan
      from ods.scd1_z_r2_ip_merchants m
      where cast(m.c_cl_org as string) in ({cft_sql_list})
    ),
    fix as (
      select cast(tt.c_tariff_plan as string) as c_tariff_plan, max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
      from ods.scd1_z_r2_tariff_tune tt
      left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
      group by cast(tt.c_tariff_plan as string)
    )
    select
      r2.r2_id,
      r2.cft_id,
      cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
      cast(dep.c_name as string) as vsp_name,
      cast(dep.c_num as string) as vsp_code,
      case
        when br.c_shortlabel is null then null
        when upper(cast(br.c_shortlabel as string)) like '%РФ%'
          then regexp_extract(cast(br.c_shortlabel as string), '^(.*?РФ)', 1)
        else cast(br.c_shortlabel as string)
      end as filial_rf,
      cast(tp.c_name as string) as tariff_name_legacy,
      cast(fix.commission_monthly_fix as decimal(18,2)) as commission_monthly_legacy
    from r2
    left join ods.scd1_z_cl_corp corp on cast(corp.id as string) = r2.cft_id
    left join ods.scd1_z_depart dep on cast(dep.id as string) = r2.c_depart
    left join ods.scd1_z_branch br on cast(br.id as string) = cast(dep.c_filial as string)
    left join ods.scd1_z_r2_tariff_plan tp on cast(tp.id as string) = r2.c_tariff_plan
    left join fix on fix.c_tariff_plan = r2.c_tariff_plan
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        r2_df = imp.fetch(sql_r2)

if r2_df is None:
    r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])

if not r2_df.empty:
    for c in ['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy']:
        r2_df[c] = r2_df[c].astype(str)
    r2_df = r2_df.drop_duplicates(subset=['cft_id'], keep='first')

print(f'R2 legacy rows: {len(r2_df):,}')
display(r2_df.head(3))

In [ ]:
# 07_base_merge
required = ['sa_df', 'cdi_map_df', 'cft_map_df', 'cmp_df', 'trx_df', 'active_term_df', 'r2_df']
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f'Сначала выполните предыдущие шаги: отсутствуют {missing}')

base_df = sa_df.copy()

if not cdi_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cdi_map_df[['inn', 'cdi_id', 'ssp_ocrm']], on='inn', how='left')
else:
    base_df['cdi_id'] = None
    base_df['ssp_ocrm'] = None

if not cft_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cft_map_df[['cdi_id', 'cft_id']], on='cdi_id', how='left')
else:
    base_df['cft_id'] = None

if not r2_df.empty and not base_df.empty:
    base_df = base_df.merge(r2_df, on='cft_id', how='left')
else:
    for col in ['r2_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy']:
        base_df[col] = None

if not cmp_df.empty and not base_df.empty:
    base_df = base_df.merge(cmp_df[['n_agr', 'retl_cnt', 'term_cnt', 'amortization']], on='n_agr', how='left')
else:
    for col in ['retl_cnt', 'term_cnt', 'amortization']:
        base_df[col] = None

if not active_term_df.empty and not base_df.empty:
    base_df = base_df.merge(active_term_df[['n_cmp_client', 'active_term_cnt']], on='n_cmp_client', how='left')
else:
    base_df['active_term_cnt'] = None

if not trx_df.empty and not base_df.empty:
    base_df = base_df.merge(trx_df[['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']], on='n_agr', how='left')
else:
    for col in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']:
        base_df[col] = None

print(f'base_df rows after merge: {len(base_df):,}')
display(base_df.head(3))

In [ ]:
# 08_agr_fallback
if 'base_df' not in globals():
    raise RuntimeError('Сначала выполните 07_base_merge')

if 'agr_id' not in base_df.columns:
    base_df['agr_id'] = None
if 'r2_id' not in base_df.columns:
    base_df['r2_id'] = None

agr_before_mask = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
base_df['agr_id_source'] = 'sa'
fallback_mask = (~agr_before_mask) & base_df['r2_id'].notna() & (base_df['r2_id'].astype(str).str.strip() != '')
base_df.loc[fallback_mask, 'agr_id'] = base_df.loc[fallback_mask, 'r2_id']
base_df.loc[fallback_mask, 'agr_id_source'] = 'r2_fallback'

filled_pct = round(100 * (base_df['agr_id'].notna().mean()), 2) if len(base_df) else 0.0
print(f'agr_id filled after fallback: {filled_pct}%')

In [ ]:
# 08b_tariff_fix_map (вынесено из 09 для переиспользования)
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров/подключения')

sql_tariff_fix_map = """
select
  cast(tt.c_tariff_plan as string) as c_tariff_plan,
  max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
from ods.scd1_z_r2_tariff_tune tt
left join ods.scd1_z_r2_tariff_fix tf
  on tt.c_tariff = tf.id
group by cast(tt.c_tariff_plan as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    tariff_fix_map_df = imp.fetch(sql_tariff_fix_map)

if tariff_fix_map_df is None:
    tariff_fix_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

if not tariff_fix_map_df.empty:
    tariff_fix_map_df['c_tariff_plan'] = tariff_fix_map_df['c_tariff_plan'].astype(str).str.strip()

print(f'tariff_fix_map rows: {len(tariff_fix_map_df):,}')
display(tariff_fix_map_df.head(3))

In [ ]:
# 09_actual_tariff_by_agr
if 'base_df' not in globals():
    raise RuntimeError('Сначала выполните 08_agr_fallback')
if 'tariff_fix_map_df' not in globals():
    raise RuntimeError('Сначала выполните 08b_tariff_fix_map')

actual_tariff_df_prev = (
    actual_tariff_df.copy()
    if 'actual_tariff_df' in globals() and isinstance(actual_tariff_df, pd.DataFrame)
    else pd.DataFrame()
)

base_df['agr_id_key'] = base_df['agr_id'].map(normalize_agr_q1)
base_agr_keys_df = (
    base_df[['agr_id_key']]
    .dropna()
    .drop_duplicates()
    .rename(columns={'agr_id_key': 'agr_id'})
)
base_agr_keys_cnt = len(base_agr_keys_df)

section09_start_ts = time.perf_counter()

if base_agr_keys_cnt:
    sql_actual_tariff_by_agr = f"""
    with agr_keys as (
      select distinct cast(a.abs_agr_id as string) as agr_id
      from ods_alpha.scd1_agreements a
      where upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
        and exists (
          select 1
          from ods_alpha.scd1_agr_terms t
          where cast(t.n_agr as string) = cast(a.n_agr as string)
            and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
            and upper(trim(cast(t.cf_ter_type as string))) = 'P'
            and coalesce(t.ods_deleted_flg, '0') <> '1'
        )
    ),
    agr_actual as (
      select
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_agr as string) as n_agr_actual,
        cast(a.c_agr_number as string) as contract_number_acq,
        cast(a.d_valid_from as date) as d_valid_from_actual,
        cast(a.d_valid_to as date) as d_valid_to_actual,
        row_number() over (
          partition by cast(a.abs_agr_id as string)
          order by cast(a.d_valid_from as date) desc, cast(a.n_agr as string) desc
        ) as rn
      from ods_alpha.scd1_agreements a
      join agr_keys k
        on k.agr_id = cast(a.abs_agr_id as string)
      where cast(a.d_valid_from as date) <= cast('{month_end}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_end}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    merchant_one as (
      select
        cast(m.id as string) as agr_id,
        cast(m.c_tariff_plan as string) as c_tariff_plan,
        row_number() over (
          partition by cast(m.id as string)
          order by cast(m.c_tariff_plan as string) desc
        ) as rn
      from ods.scd1_z_r2_ip_merchants m
      join agr_keys k
        on k.agr_id = cast(m.id as string)
    )
    select
      m.agr_id,
      aa.n_agr_actual,
      aa.contract_number_acq,
      aa.d_valid_from_actual,
      aa.d_valid_to_actual,
      m.c_tariff_plan,
      cast(tp.c_name as string) as tariff_name_actual
    from merchant_one m
    left join agr_actual aa
      on aa.agr_id = m.agr_id
     and aa.rn = 1
    left join ods.scd1_z_r2_tariff_plan tp
      on cast(tp.id as string) = m.c_tariff_plan
    where m.rn = 1
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        actual_tariff_df = imp.fetch(sql_actual_tariff_by_agr)
else:
    actual_tariff_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'c_tariff_plan', 'tariff_name_actual'
    ])

if actual_tariff_df is None:
    actual_tariff_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'c_tariff_plan', 'tariff_name_actual'
    ])

if not actual_tariff_df.empty:
    for c in ['agr_id', 'n_agr_actual', 'contract_number_acq', 'c_tariff_plan', 'tariff_name_actual']:
        actual_tariff_df[c] = actual_tariff_df[c].astype(str).str.strip()

    # Ограничиваем результат ключами текущего base_df (сохраняем ту же прикладную выборку)
    actual_tariff_df = actual_tariff_df.merge(base_agr_keys_df, on='agr_id', how='inner')

actual_tariff_df = actual_tariff_df.rename(columns={'agr_id': 'agr_id_key'})

if 'c_tariff_plan' not in actual_tariff_df.columns:
    actual_tariff_df['c_tariff_plan'] = None

if not actual_tariff_df.empty and not tariff_fix_map_df.empty:
    actual_tariff_df = actual_tariff_df.merge(tariff_fix_map_df, on='c_tariff_plan', how='left')
else:
    actual_tariff_df['commission_monthly_fix'] = None

actual_tariff_df['commission_monthly_actual'] = actual_tariff_df.get('commission_monthly_fix')
if 'commission_monthly_fix' in actual_tariff_df.columns:
    actual_tariff_df = actual_tariff_df.drop(columns=['commission_monthly_fix'])

section09_elapsed_sec = round(time.perf_counter() - section09_start_ts, 2)

# QC эквивалентности с предыдущим результатом (если был в памяти)
if isinstance(actual_tariff_df_prev, pd.DataFrame) and len(actual_tariff_df_prev):
    prev_cmp_df = actual_tariff_df_prev[['agr_id_key', 'tariff_name_actual']].copy() if 'agr_id_key' in actual_tariff_df_prev.columns and 'tariff_name_actual' in actual_tariff_df_prev.columns else pd.DataFrame(columns=['agr_id_key', 'tariff_name_actual'])
    new_cmp_df = actual_tariff_df[['agr_id_key', 'tariff_name_actual']].copy() if 'agr_id_key' in actual_tariff_df.columns and 'tariff_name_actual' in actual_tariff_df.columns else pd.DataFrame(columns=['agr_id_key', 'tariff_name_actual'])

    prev_cmp_df = prev_cmp_df.drop_duplicates(subset=['agr_id_key'])
    new_cmp_df = new_cmp_df.drop_duplicates(subset=['agr_id_key'])

    eq_cmp_df = prev_cmp_df.merge(new_cmp_df, on='agr_id_key', how='outer', suffixes=('_prev', '_new'), indicator='src')
    eq_cmp_df['_prev_norm'] = eq_cmp_df['tariff_name_actual_prev'].fillna('').astype(str).str.strip().str.lower()
    eq_cmp_df['_new_norm'] = eq_cmp_df['tariff_name_actual_new'].fillna('').astype(str).str.strip().str.lower()

    tariff_changed_cnt = int(((eq_cmp_df['src'] == 'both') & (eq_cmp_df['_prev_norm'] != eq_cmp_df['_new_norm'])).sum())
    only_prev_cnt = int((eq_cmp_df['src'] == 'left_only').sum())
    only_new_cnt = int((eq_cmp_df['src'] == 'right_only').sum())

    print('QC эквивалентности (prev vs new):')
    print(f'changed_tariff_name: {tariff_changed_cnt:,}')
    print(f'only_prev_keys: {only_prev_cnt:,}')
    print(f'only_new_keys: {only_new_cnt:,}')

print(f'base agr_id keys: {base_agr_keys_cnt:,}')
print(f'actual_tariff rows: {len(actual_tariff_df):,}')
print(f'section_09_elapsed_sec: {section09_elapsed_sec}')
display(actual_tariff_df.head(3))

In [ ]:
# 10_apply_tariff_fix_and_formulas -> final_df
if 'base_df' not in globals() or 'actual_tariff_df' not in globals():
    raise RuntimeError('Сначала выполните 09_actual_tariff_by_agr')

# Совместимость с предыдущими версиями 09 (где имена колонок могли отличаться)
if 'tariff_name_actual' not in actual_tariff_df.columns and 'tariff_name' in actual_tariff_df.columns:
    actual_tariff_df['tariff_name_actual'] = actual_tariff_df['tariff_name']
if 'commission_monthly_actual' not in actual_tariff_df.columns and 'commission_monthly' in actual_tariff_df.columns:
    actual_tariff_df['commission_monthly_actual'] = actual_tariff_df['commission_monthly']

required_tariff_cols = [
    'agr_id_key', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual',
    'd_valid_to_actual', 'c_tariff_plan', 'tariff_name_actual', 'commission_monthly_actual'
]
for col in required_tariff_cols:
    if col not in actual_tariff_df.columns:
        actual_tariff_df[col] = None

# Идемпотентность: при повторном запуске 10-й секции убираем старые колонки и суффиксы _x/_y
merge_payload_cols = [c for c in required_tariff_cols if c != 'agr_id_key']
drop_before_merge = []
for c in merge_payload_cols:
    for c_try in [c, f'{c}_x', f'{c}_y']:
        if c_try in base_df.columns:
            drop_before_merge.append(c_try)
if drop_before_merge:
    base_df = base_df.drop(columns=sorted(set(drop_before_merge)))

base_df = base_df.merge(
    actual_tariff_df[required_tariff_cols],
    on='agr_id_key',
    how='left'
)

base_df['tariff_name'] = base_df['tariff_name_actual'].where(
    base_df['tariff_name_actual'].notna() & (base_df['tariff_name_actual'].astype(str).str.strip() != ''),
    base_df['tariff_name_legacy']
)
base_df['tariff_source'] = base_df['tariff_name_actual'].apply(
    lambda x: 'agr_id_actual' if pd.notna(x) and str(x).strip() else 'legacy_cft'
)
base_df['commission_monthly'] = pd.to_numeric(base_df['commission_monthly_actual'], errors='coerce').where(
    pd.to_numeric(base_df['commission_monthly_actual'], errors='coerce').notna(),
    pd.to_numeric(base_df['commission_monthly_legacy'], errors='coerce')
)

commission_from_ops_num = pd.to_numeric(base_df.get('commission_from_ops'), errors='coerce').fillna(0)
commission_monthly_num = pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0)
int_component_num = pd.to_numeric(base_df.get('int_component'), errors='coerce').fillna(0)
amortization_num = pd.to_numeric(base_df.get('amortization'), errors='coerce').fillna(0)
retl_cnt_num = pd.to_numeric(base_df.get('retl_cnt'), errors='coerce').fillna(0)

base_df['commission_total'] = commission_from_ops_num + commission_monthly_num
base_df['aur'] = retl_cnt_num * 1926
base_df['chod'] = base_df['commission_total'] + int_component_num
base_df['fin_result'] = base_df['chod'] - pd.to_numeric(base_df['aur'], errors='coerce').fillna(0) - amortization_num
base_df['report_month'] = report_month_label
base_df['snapshot_month_start'] = snapshot_month_start

mvp_columns = [
    'report_month', 'snapshot_month_start', 'inn', 'company_name',
    'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
    'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
    'cdi_id', 'ssp_ocrm', 'cft_id', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code',
    'tariff_name', 'tariff_source',
    'retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
    'aur', 'amortization', 'chod', 'fin_result'
]

for col in mvp_columns:
    if col not in base_df.columns:
        base_df[col] = None

for col in ['trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total', 'aur', 'amortization', 'chod', 'fin_result']:
    base_df[col] = base_df[col].map(to_decimal_or_none)

for col in ['retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt']:
    base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype('Int64')

final_df = base_df[mvp_columns].copy()

if final_df is None or len(final_df) == 0:
    raise RuntimeError('QC fail: final_df пустой')
if 'tariff_name' not in final_df.columns:
    raise RuntimeError('QC fail: отсутствует tariff_name')

tariff_actual_rows = int((final_df['tariff_source'] == 'agr_id_actual').sum()) if len(final_df) else 0
tariff_actual_pct = round(100 * tariff_actual_rows / len(final_df), 2) if len(final_df) else 0.0

print(f'final_df rows: {len(final_df):,}')
print(f'tariff_source=agr_id_actual: {tariff_actual_rows:,} ({tariff_actual_pct}%)')
display(final_df.head(5))

## Секция 2. Сравнение `final_df` и Excel (`cmp` + KPI)

Секция независима от шагов выше, кроме наличия `final_df`.

In [ ]:
# Compare + KPI (ключевые метрики из запроса)
if 'final_df' not in globals():
    raise RuntimeError('Сначала выполните 10_apply_tariff_fix_and_formulas')
for key_col in ['inn', 'agr_id']:
    if key_col not in final_df.columns:
        raise RuntimeError(f'В final_df отсутствует ключевая колонка: {key_col}')

if not run_excel_compare:
    cmp = pd.DataFrame()
    kpi_compare_df = pd.DataFrame()
    totals_df = pd.DataFrame()
    print('Excel compare skipped (run_excel_compare=False)')
else:
    ex = pd.read_excel(excel_path, header=excel_header)

    # Маппинг колонок Excel под ключевые метрики
    col_map = {
        'inn_col': ['ИНН', 'inn', 'c_inn'],
        'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
        'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
        'term_col': ['Кол-во терминалов', 'Количество терминалов'],
        'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
        'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
        'comm_total_col': ['Комиссия эквайринга'],
        'int_component_col': ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)'],
        'comm_monthly_col': ['Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)', 'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)'],
        'chod_col': ['ЧОД'],
        'aur_col': ['АУР', 'AUR', 'Aur'],
        'fin_result_col': ['Фин. Рез.', 'Фин.Рез.', 'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат', 'fin_result'],
    }

    resolved = {k: pick_col_robust(ex.columns, v) for k, v in col_map.items()}
    missing = [k for k, v in resolved.items() if v is None]
    if missing:
        raise ValueError(f'Не найдены колонки Excel: {missing}. Доступные: {list(ex.columns)}')

    print('Используемые колонки Excel:')
    display(pd.DataFrame([{'logical_name': k, 'excel_column': v} for k, v in resolved.items()]))

    ex['inn_key'] = ex[resolved['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved['agr_col']].apply(normalize_agr_q1)

    ex['retl_cnt_excel'] = pd.to_numeric(ex[resolved['retl_col']], errors='coerce')
    ex['term_cnt_excel'] = pd.to_numeric(ex[resolved['term_col']], errors='coerce')
    ex['trx_cnt_excel'] = pd.to_numeric(ex[resolved['trx_cnt_col']], errors='coerce')
    ex['trx_sum_excel'] = to_num_series(ex[resolved['trx_sum_col']])
    ex['commission_total_excel'] = to_num_series(ex[resolved['comm_total_col']])
    ex['int_component_excel'] = to_num_series(ex[resolved['int_component_col']])
    ex['commission_monthly_excel'] = to_num_series(ex[resolved['comm_monthly_col']])
    ex['chod_excel'] = to_num_series(ex[resolved['chod_col']])
    ex['aur_excel'] = to_num_series(ex[resolved['aur_col']])
    ex['fin_result_excel'] = to_num_series(ex[resolved['fin_result_col']])

    ex_agg = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_excel': 'max',
              'term_cnt_excel': 'max',
              'trx_cnt_excel': 'max',
              'trx_sum_excel': 'sum',
              'commission_total_excel': 'sum',
              'int_component_excel': 'sum',
              'commission_monthly_excel': 'max',
              'chod_excel': 'sum',
              'aur_excel': 'sum',
              'fin_result_excel': 'sum',
          })
    )

    lk = final_df.copy()
    lk['inn_key'] = lk['inn'].apply(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].apply(normalize_agr_q1)

    lk['retl_cnt_lake'] = pd.to_numeric(lk['retl_cnt'], errors='coerce')
    lk['term_cnt_lake'] = pd.to_numeric(lk['term_cnt'], errors='coerce')
    lk['trx_cnt_lake'] = pd.to_numeric(lk['trx_cnt'], errors='coerce')
    lk['trx_sum_lake'] = pd.to_numeric(lk['trx_sum'], errors='coerce')
    lk['commission_total_lake'] = pd.to_numeric(lk['commission_total'], errors='coerce')
    lk['int_component_lake'] = pd.to_numeric(lk['int_component'], errors='coerce')
    lk['commission_monthly_lake'] = pd.to_numeric(lk['commission_monthly'], errors='coerce')
    lk['chod_lake'] = pd.to_numeric(lk['chod'], errors='coerce')
    lk['aur_lake'] = pd.to_numeric(lk['aur'], errors='coerce')
    lk['fin_result_lake'] = pd.to_numeric(lk['fin_result'], errors='coerce')

    lk_agg = (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_lake': 'max',
              'term_cnt_lake': 'max',
              'trx_cnt_lake': 'max',
              'trx_sum_lake': 'max',
              'commission_total_lake': 'max',
              'int_component_lake': 'max',
              'commission_monthly_lake': 'max',
              'chod_lake': 'max',
              'aur_lake': 'max',
              'fin_result_lake': 'max',
          })
    )

    cmp = lk_agg.merge(ex_agg, on=['inn_key', 'agr_id_key'], how='inner')

    metric_keys = [
        'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
        'commission_total', 'int_component', 'commission_monthly',
        'chod', 'aur', 'fin_result'
    ]

    for m in metric_keys:
        cmp[f'{m}_delta'] = cmp[f'{m}_lake'].fillna(0) - cmp[f'{m}_excel'].fillna(0)
        cmp[f'{m}_abs'] = cmp[f'{m}_delta'].abs()

    # KPI
    count_metrics = ['retl_cnt', 'term_cnt', 'trx_cnt']
    money_metrics = ['trx_sum', 'commission_total', 'int_component', 'commission_monthly', 'chod', 'aur', 'fin_result']

    kpi_rows = [{'metric': 'rows_compared', 'value': len(cmp)}]
    for m in count_metrics:
        kpi_rows.append({'metric': f'{m}_exact_match_rate_pct_tol_0', 'value': exact_match_rate_from_abs(cmp[f'{m}_abs'], tol=0.0)})
        kpi_rows.append({'metric': f'{m}_mae', 'value': float(cmp[f'{m}_abs'].mean()) if len(cmp) else 0.0})
    for m in money_metrics:
        kpi_rows.append({'metric': f'{m}_exact_match_rate_pct_tol_0_01', 'value': exact_match_rate_from_abs(cmp[f'{m}_abs'], tol=0.01)})
        kpi_rows.append({'metric': f'{m}_mae', 'value': float(cmp[f'{m}_abs'].mean()) if len(cmp) else 0.0})

    kpi_compare_df = pd.DataFrame(kpi_rows)

    totals_row = {'report_month': report_month, 'rows_compared': len(cmp)}
    for m in metric_keys:
        totals_row[f'excel_{m}_total'] = cmp[f'{m}_excel'].fillna(0).sum()
        totals_row[f'lake_{m}_total'] = cmp[f'{m}_lake'].fillna(0).sum()
        totals_row[f'delta_{m}_total'] = totals_row[f'lake_{m}_total'] - totals_row[f'excel_{m}_total']
    totals_df = pd.DataFrame([totals_row])

    if cmp is None or len(cmp) == 0:
        raise RuntimeError('QC fail: cmp не сформирован или пустой')

    print(f'cmp rows: {len(cmp):,}')
    display(kpi_compare_df)
    display(totals_df)

    if show_compare_top:
        for m in metric_keys:
            cols = ['agr_id_key', 'inn_key', f'{m}_lake', f'{m}_excel', f'{m}_delta']
            print(f'TOP-10 {m} by abs delta')
            display(cmp.sort_values(f'{m}_abs', ascending=False)[cols].head(10))

if save_final_csv:
    out = Path(output_csv_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(out, index=False, encoding='utf-8-sig')
    print(f'final_df exported to: {out.resolve()}')

## Секция 3. Диагностика расхождений `retl_cnt` и `term_cnt`

Секция помогает понять источник расхождений между Excel и Озером:
- пустые/пропущенные значения в Lake;
- крупные дельты по `agr_id`;
- дубликаты договоров в Lake по ключу `inn+agr_id`;
- связь расхождений с периметром `merchant/terminal`.

In [ ]:
# 3.1 Базовая диагностика по cmp
if 'cmp' not in globals() or cmp is None or len(cmp) == 0:
    raise RuntimeError('Сначала выполните compare-секцию (нужен DataFrame cmp).')
if 'base_df' not in globals() or base_df is None or len(base_df) == 0:
    raise RuntimeError('Сначала выполните секции до 10 (нужен DataFrame base_df).')

diag_top_n = 20

# Контроль NA/пропусков в Lake относительно Excel
retl_na_issue_df = cmp[cmp['retl_cnt_lake'].isna() & cmp['retl_cnt_excel'].notna()].copy()
term_na_issue_df = cmp[cmp['term_cnt_lake'].isna() & cmp['term_cnt_excel'].notna()].copy()

print(f"retl: Lake NA при наличии Excel = {len(retl_na_issue_df):,}")
print(f"term: Lake NA при наличии Excel = {len(term_na_issue_df):,}")

# TOP расхождений
retl_top_abs_df = cmp.sort_values('retl_cnt_abs', ascending=False).head(diag_top_n).copy()
term_top_abs_df = cmp.sort_values('term_cnt_abs', ascending=False).head(diag_top_n).copy()

print(f'\nTOP-{diag_top_n} retl_cnt by abs delta:')
display(retl_top_abs_df[['agr_id_key', 'inn_key', 'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'retl_cnt_abs']])

print(f'TOP-{diag_top_n} term_cnt by abs delta:')
display(term_top_abs_df[['agr_id_key', 'inn_key', 'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta', 'term_cnt_abs']])

# Ключи для углубленной диагностики
focus_keys_df = pd.concat([
    retl_top_abs_df[['agr_id_key', 'inn_key']],
    term_top_abs_df[['agr_id_key', 'inn_key']]
], ignore_index=True).drop_duplicates().reset_index(drop=True)

# Диагностика дубликатов на стороне Lake (по base_df)
base_keys_df = base_df.copy()
base_keys_df['agr_id_key'] = base_keys_df['agr_id'].apply(normalize_agr_q1)
base_keys_df['inn_key'] = base_keys_df['inn'].apply(normalize_inn_q1)

dup_lake_keys_df = (
    base_keys_df.dropna(subset=['agr_id_key', 'inn_key'])
      .groupby(['agr_id_key', 'inn_key'], as_index=False)
      .agg(rows=('agr_id_key', 'size'), n_agr_nunique=('n_agr', 'nunique'))
      .query('rows > 1 or n_agr_nunique > 1')
      .sort_values(['rows', 'n_agr_nunique'], ascending=False)
)

print(f'\nКлючей inn+agr_id с дублями в base_df: {len(dup_lake_keys_df):,}')
if not dup_lake_keys_df.empty:
    display(dup_lake_keys_df.head(20))

focus_detail_df = (
    focus_keys_df
    .merge(cmp[['agr_id_key', 'inn_key', 'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'retl_cnt_abs',
                'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta', 'term_cnt_abs']],
           on=['agr_id_key', 'inn_key'], how='left')
    .merge(base_keys_df[['agr_id_key', 'inn_key', 'n_agr', 'n_cmp_client', 'company_name']].drop_duplicates(),
           on=['agr_id_key', 'inn_key'], how='left')
)

print(f'\nFocus keys для SQL-диагностики: {len(focus_detail_df):,}')
display(focus_detail_df.sort_values(['retl_cnt_abs', 'term_cnt_abs'], ascending=False).head(50))

In [ ]:
# 3.2 SQL-диагностика периметра merchant/terminal по проблемным n_agr
if 'focus_detail_df' not in globals() or focus_detail_df is None or len(focus_detail_df) == 0:
    raise RuntimeError('Сначала выполните 3.1 (нужен DataFrame focus_detail_df).')

focus_n_agr_values = sorted([x for x in focus_detail_df['n_agr'].dropna().astype(str).unique().tolist() if x])

if not focus_n_agr_values:
    print('Для focus-ключей не найден n_agr. Пропускаем SQL-диагностику.')
    perimeter_diag_df = pd.DataFrame()
else:
    focus_n_agr_sql = ', '.join([f"'{x}'" for x in focus_n_agr_values])

    sql_perimeter_diag = f"""
    with sa_agr as (
      select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.n_cmp_client as string) as n_cmp_client,
        cast(a.abs_agr_id as string) as agr_id
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({focus_n_agr_sql})
    ),
    m_all as (
      select
        sa.n_agr,
        sa.n_cmp_client,
        cast(m.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants m
        on cast(m.n_cmp as string) = sa.n_cmp_client
      where m.c_nmrc is not null
        and coalesce(m.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(m.c_nmrc as string)
    ),
    term_all as (
      select
        ma.n_agr,
        ma.n_cmp_client,
        ma.c_nmrc,
        cast(t.c_nter as string) as c_nter
      from m_all ma
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = ma.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
      group by ma.n_agr, ma.n_cmp_client, ma.c_nmrc, cast(t.c_nter as string)
    ),
    term_active_month as (
      select
        ma.n_agr,
        ma.n_cmp_client,
        ma.c_nmrc,
        cast(t.c_nter as string) as c_nter
      from m_all ma
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = ma.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
      group by ma.n_agr, ma.n_cmp_client, ma.c_nmrc, cast(t.c_nter as string)
    ),
    retl_all as (
      select n_agr, count(distinct c_nmrc) as retl_all_cnt
      from m_all
      group by n_agr
    ),
    retl_active as (
      select n_agr, count(distinct c_nmrc) as retl_active_cnt
      from term_active_month
      group by n_agr
    ),
    term_all_agg as (
      select n_agr, count(distinct c_nter) as term_all_cnt
      from term_all
      group by n_agr
    ),
    term_active_agg as (
      select n_agr, count(distinct c_nter) as term_active_cnt
      from term_active_month
      group by n_agr
    )
    select
      sa.agr_id,
      sa.n_agr,
      sa.n_cmp_client,
      ra.retl_all_cnt,
      rv.retl_active_cnt,
      ta.term_all_cnt,
      tv.term_active_cnt
    from sa_agr sa
    left join retl_all ra on ra.n_agr = sa.n_agr
    left join retl_active rv on rv.n_agr = sa.n_agr
    left join term_all_agg ta on ta.n_agr = sa.n_agr
    left join term_active_agg tv on tv.n_agr = sa.n_agr
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        perimeter_diag_df = imp.fetch(sql_perimeter_diag)

    if perimeter_diag_df is None:
        perimeter_diag_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'retl_all_cnt', 'retl_active_cnt', 'term_all_cnt', 'term_active_cnt'])

print(f'perimeter_diag rows: {len(perimeter_diag_df):,}')

if len(perimeter_diag_df):
    perimeter_diag_df['agr_id_key'] = perimeter_diag_df['agr_id'].apply(normalize_agr_q1)
    perimeter_focus_cmp_df = focus_detail_df.merge(perimeter_diag_df, on=['agr_id_key', 'n_agr', 'n_cmp_client'], how='left')

    # Восстанавливаем дельты/abs при частичных перезапусках (чтобы 3.2 была self-healing)
    for m in ['retl_cnt', 'term_cnt']:
        lake_col = f'{m}_lake'
        excel_col = f'{m}_excel'
        delta_col = f'{m}_delta'
        abs_col = f'{m}_abs'

        if delta_col not in perimeter_focus_cmp_df.columns:
            perimeter_focus_cmp_df[delta_col] = (
                pd.to_numeric(perimeter_focus_cmp_df.get(lake_col), errors='coerce').fillna(0)
                - pd.to_numeric(perimeter_focus_cmp_df.get(excel_col), errors='coerce').fillna(0)
            )
        if abs_col not in perimeter_focus_cmp_df.columns:
            perimeter_focus_cmp_df[abs_col] = pd.to_numeric(perimeter_focus_cmp_df[delta_col], errors='coerce').abs()

    # Удобные диагностические флаги
    perimeter_focus_cmp_df['flag_lake_retl_missing'] = perimeter_focus_cmp_df['retl_cnt_lake'].isna() & perimeter_focus_cmp_df['retl_cnt_excel'].notna()
    perimeter_focus_cmp_df['flag_excel_gt_retl_active'] = perimeter_focus_cmp_df['retl_cnt_excel'].fillna(0) > perimeter_focus_cmp_df['retl_active_cnt'].fillna(0)
    perimeter_focus_cmp_df['flag_excel_gt_term_active'] = perimeter_focus_cmp_df['term_cnt_excel'].fillna(0) > perimeter_focus_cmp_df['term_active_cnt'].fillna(0)

    print('Диагностика причин по focus-ключам:')
    display_cols = [
        'agr_id_key', 'inn_key', 'n_agr', 'company_name',
        'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'retl_cnt_abs',
        'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta', 'term_cnt_abs',
        'retl_all_cnt', 'retl_active_cnt', 'term_all_cnt', 'term_active_cnt',
        'flag_lake_retl_missing', 'flag_excel_gt_retl_active', 'flag_excel_gt_term_active'
    ]
    display_cols = [c for c in display_cols if c in perimeter_focus_cmp_df.columns]

    display_df = perimeter_focus_cmp_df[display_cols].copy()
    sort_cols = [c for c in ['retl_cnt_abs', 'term_cnt_abs'] if c in display_df.columns]
    if sort_cols:
        display_df = display_df.sort_values(sort_cols, ascending=False)

    display(display_df.head(100))
else:
    print('Нет данных SQL-диагностики: проверьте focus_n_agr_values и права на таблицы.')

### 3.3 Проверка гипотезы: считать `retl_cnt`/`term_cnt` без фильтра активных терминалов

Сравниваем для focus-ключей два подхода:
- `active` (текущая логика из Lake),
- `all` (без фильтра по датам установки/закрытия терминалов).

Если `all` системно ближе к Excel, значит причина расхождений действительно в фильтре `term_active`.

In [ ]:
# 3.3 Сравнение quality: active vs all (без фильтра активных терминалов)
if 'perimeter_focus_cmp_df' not in globals() or perimeter_focus_cmp_df is None or len(perimeter_focus_cmp_df) == 0:
    raise RuntimeError('Сначала выполните 3.2 (нужен DataFrame perimeter_focus_cmp_df).')

hyp_df = perimeter_focus_cmp_df.copy()

num_cols = [
    'retl_cnt_excel', 'retl_active_cnt', 'retl_all_cnt',
    'term_cnt_excel', 'term_active_cnt', 'term_all_cnt'
]
for c in num_cols:
    if c not in hyp_df.columns:
        hyp_df[c] = pd.NA
    hyp_df[c] = pd.to_numeric(hyp_df[c], errors='coerce')

# Абсолютные дельты к Excel
hyp_df['retl_abs_active'] = (hyp_df['retl_cnt_excel'] - hyp_df['retl_active_cnt']).abs()
hyp_df['retl_abs_all'] = (hyp_df['retl_cnt_excel'] - hyp_df['retl_all_cnt']).abs()
hyp_df['term_abs_active'] = (hyp_df['term_cnt_excel'] - hyp_df['term_active_cnt']).abs()
hyp_df['term_abs_all'] = (hyp_df['term_cnt_excel'] - hyp_df['term_all_cnt']).abs()

# Где вариант "без active-фильтра" лучше
hyp_df['retl_better_without_active'] = hyp_df['retl_abs_all'] < hyp_df['retl_abs_active']
hyp_df['term_better_without_active'] = hyp_df['term_abs_all'] < hyp_df['term_abs_active']

# Величина улучшения (положительное = all лучше)
hyp_df['retl_improvement_abs'] = hyp_df['retl_abs_active'] - hyp_df['retl_abs_all']
hyp_df['term_improvement_abs'] = hyp_df['term_abs_active'] - hyp_df['term_abs_all']

summary_rows = [
    {
        'metric': 'retl_cnt',
        'rows_total': int(len(hyp_df)),
        'better_without_active_cnt': int(hyp_df['retl_better_without_active'].fillna(False).sum()),
        'better_without_active_pct': round(100 * hyp_df['retl_better_without_active'].fillna(False).mean(), 2),
        'mae_active': round(float(hyp_df['retl_abs_active'].mean()), 4) if len(hyp_df) else 0.0,
        'mae_all': round(float(hyp_df['retl_abs_all'].mean()), 4) if len(hyp_df) else 0.0,
    },
    {
        'metric': 'term_cnt',
        'rows_total': int(len(hyp_df)),
        'better_without_active_cnt': int(hyp_df['term_better_without_active'].fillna(False).sum()),
        'better_without_active_pct': round(100 * hyp_df['term_better_without_active'].fillna(False).mean(), 2),
        'mae_active': round(float(hyp_df['term_abs_active'].mean()), 4) if len(hyp_df) else 0.0,
        'mae_all': round(float(hyp_df['term_abs_all'].mean()), 4) if len(hyp_df) else 0.0,
    }
]

hypothesis_summary_df = pd.DataFrame(summary_rows)
print('Гипотеза: вариант без фильтра active ближе к Excel')
display(hypothesis_summary_df)

print('\nTOP кейсов, где all заметно лучше для retl_cnt:')
display(
    hyp_df.sort_values('retl_improvement_abs', ascending=False)[[
        'agr_id_key', 'inn_key', 'company_name',
        'retl_cnt_excel', 'retl_active_cnt', 'retl_all_cnt',
        'retl_abs_active', 'retl_abs_all', 'retl_improvement_abs'
    ]].head(30)
)

print('TOP кейсов, где all заметно лучше для term_cnt:')
display(
    hyp_df.sort_values('term_improvement_abs', ascending=False)[[
        'agr_id_key', 'inn_key', 'company_name',
        'term_cnt_excel', 'term_active_cnt', 'term_all_cnt',
        'term_abs_active', 'term_abs_all', 'term_improvement_abs'
    ]].head(30)
)

# Доп. срез: где текущая Lake-логика сильно ниже Excel и all закрывает разрыв
retl_recovered_df = hyp_df[
    (hyp_df['retl_cnt_excel'].fillna(0) > hyp_df['retl_active_cnt'].fillna(0))
    & (hyp_df['retl_all_cnt'].fillna(0) >= hyp_df['retl_cnt_excel'].fillna(0))
].copy()

print(f"Кейсов, где retl_active < Excel, но retl_all >= Excel: {len(retl_recovered_df):,}")
if not retl_recovered_df.empty:
    display(retl_recovered_df[['agr_id_key', 'inn_key', 'company_name', 'retl_cnt_excel', 'retl_active_cnt', 'retl_all_cnt']].head(30))

### 3.4 Проверка гипотезы: расхождения из-за нескольких `n_agr` внутри одного `agr_id`

Секция проверяет, насколько текущая агрегация Lake по ключу `inn+agr_id` через `max` может давать расхождения, когда в месяце у одного `agr_id` несколько версий договора (`n_agr`).

Сравниваются два варианта агрегации Lake для таких ключей:
- `max` (как сейчас),
- `sum` (альтернативная гипотеза).

Далее считается, какой вариант ближе к Excel по абсолютной дельте.

In [ ]:
# 3.4 multi n_agr hypothesis check: lake max vs lake sum
if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала выполните 10_apply_tariff_fix_and_formulas (нужен final_df).')

# Нужны те же ключи и метрики, что в compare
metrics_check = [
    'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
    'commission_total', 'int_component', 'commission_monthly',
    'chod', 'aur', 'fin_result'
]

lk_base = final_df.copy()
lk_base['inn_key'] = lk_base['inn'].apply(normalize_inn_q1)
lk_base['agr_id_key'] = lk_base['agr_id'].apply(normalize_agr_q1)
lk_base['n_agr_key'] = lk_base['n_agr'].astype(str)

for m in metrics_check:
    lk_base[f'{m}_lake'] = pd.to_numeric(lk_base[m], errors='coerce')

lk_base = lk_base.dropna(subset=['inn_key', 'agr_id_key'])

# Ключи, где в Lake больше 1 n_agr внутри месяца
multi_nagr_keys_df = (
    lk_base.groupby(['inn_key', 'agr_id_key'], as_index=False)
           .agg(n_agr_nunique=('n_agr_key', 'nunique'), rows=('n_agr_key', 'size'))
           .query('n_agr_nunique > 1')
           .sort_values(['n_agr_nunique', 'rows'], ascending=False)
)

print(f"Ключей inn+agr_id с несколькими n_agr: {len(multi_nagr_keys_df):,}")
if not multi_nagr_keys_df.empty:
    display(multi_nagr_keys_df.head(20))

if multi_nagr_keys_df.empty:
    print('Гипотеза по multi n_agr не подтверждается на текущем наборе: таких ключей нет.')
else:
    # Excel aggregation (как в compare-секции)
    ex_local = pd.read_excel(excel_path, header=excel_header)
    col_map_local = {
        'inn_col': ['ИНН', 'inn', 'c_inn'],
        'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
        'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
        'term_col': ['Кол-во терминалов', 'Количество терминалов'],
        'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
        'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
        'comm_total_col': ['Комиссия эквайринга'],
        'int_component_col': ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)'],
        'comm_monthly_col': ['Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)', 'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)'],
        'chod_col': ['ЧОД'],
        'aur_col': ['АУР', 'AUR', 'Aur'],
        'fin_result_col': ['Фин. Рез.', 'Фин.Рез.', 'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат', 'fin_result'],
    }
    resolved_local = {k: pick_col_robust(ex_local.columns, v) for k, v in col_map_local.items()}
    missing_local = [k for k, v in resolved_local.items() if v is None]
    if missing_local:
        raise ValueError(f'Для 3.4 не найдены колонки Excel: {missing_local}')

    ex_local['inn_key'] = ex_local[resolved_local['inn_col']].apply(normalize_inn_q1)
    ex_local['agr_id_key'] = ex_local[resolved_local['agr_col']].apply(normalize_agr_q1)

    ex_local['retl_cnt_excel'] = pd.to_numeric(ex_local[resolved_local['retl_col']], errors='coerce')
    ex_local['term_cnt_excel'] = pd.to_numeric(ex_local[resolved_local['term_col']], errors='coerce')
    ex_local['trx_cnt_excel'] = pd.to_numeric(ex_local[resolved_local['trx_cnt_col']], errors='coerce')
    ex_local['trx_sum_excel'] = to_num_series(ex_local[resolved_local['trx_sum_col']])
    ex_local['commission_total_excel'] = to_num_series(ex_local[resolved_local['comm_total_col']])
    ex_local['int_component_excel'] = to_num_series(ex_local[resolved_local['int_component_col']])
    ex_local['commission_monthly_excel'] = to_num_series(ex_local[resolved_local['comm_monthly_col']])
    ex_local['chod_excel'] = to_num_series(ex_local[resolved_local['chod_col']])
    ex_local['aur_excel'] = to_num_series(ex_local[resolved_local['aur_col']])
    ex_local['fin_result_excel'] = to_num_series(ex_local[resolved_local['fin_result_col']])

    ex_local_agg = (
        ex_local.dropna(subset=['inn_key', 'agr_id_key'])
                .groupby(['inn_key', 'agr_id_key'], as_index=False)
                .agg({
                    'retl_cnt_excel': 'max',
                    'term_cnt_excel': 'max',
                    'trx_cnt_excel': 'max',
                    'trx_sum_excel': 'sum',
                    'commission_total_excel': 'sum',
                    'int_component_excel': 'sum',
                    'commission_monthly_excel': 'max',
                    'chod_excel': 'sum',
                    'aur_excel': 'sum',
                    'fin_result_excel': 'sum',
                })
    )

    # Оставляем только multi-keys
    lk_multi = lk_base.merge(multi_nagr_keys_df[['inn_key', 'agr_id_key']], on=['inn_key', 'agr_id_key'], how='inner')

    agg_map_max = {f'{m}_lake': 'max' for m in metrics_check}
    agg_map_sum = {f'{m}_lake': 'sum' for m in metrics_check}

    lk_multi_max = lk_multi.groupby(['inn_key', 'agr_id_key'], as_index=False).agg(agg_map_max)
    lk_multi_sum = lk_multi.groupby(['inn_key', 'agr_id_key'], as_index=False).agg(agg_map_sum)

    cmp_multi_max = lk_multi_max.merge(ex_local_agg, on=['inn_key', 'agr_id_key'], how='inner')
    cmp_multi_sum = lk_multi_sum.merge(ex_local_agg, on=['inn_key', 'agr_id_key'], how='inner')

    for m in metrics_check:
        cmp_multi_max[f'{m}_abs'] = (cmp_multi_max[f'{m}_lake'] - cmp_multi_max[f'{m}_excel']).abs()
        cmp_multi_sum[f'{m}_abs'] = (cmp_multi_sum[f'{m}_lake'] - cmp_multi_sum[f'{m}_excel']).abs()

    summary_rows = []
    for m in metrics_check:
        mae_max = float(cmp_multi_max[f'{m}_abs'].mean()) if len(cmp_multi_max) else 0.0
        mae_sum = float(cmp_multi_sum[f'{m}_abs'].mean()) if len(cmp_multi_sum) else 0.0

        pair_df = cmp_multi_max[['inn_key', 'agr_id_key', f'{m}_abs']].merge(
            cmp_multi_sum[['inn_key', 'agr_id_key', f'{m}_abs']],
            on=['inn_key', 'agr_id_key'],
            suffixes=('_max', '_sum')
        )
        better_sum_cnt = int((pair_df[f'{m}_abs_sum'] < pair_df[f'{m}_abs_max']).sum())
        better_max_cnt = int((pair_df[f'{m}_abs_max'] < pair_df[f'{m}_abs_sum']).sum())
        equal_cnt = int((pair_df[f'{m}_abs_max'] == pair_df[f'{m}_abs_sum']).sum())

        summary_rows.append({
            'metric': m,
            'rows_compared_multi_keys': int(len(pair_df)),
            'mae_lake_max': round(mae_max, 4),
            'mae_lake_sum': round(mae_sum, 4),
            'better_sum_cnt': better_sum_cnt,
            'better_max_cnt': better_max_cnt,
            'equal_cnt': equal_cnt,
        })

    multi_nagr_hypothesis_df = pd.DataFrame(summary_rows)

    print('Итог гипотезы (multi n_agr: max vs sum):')
    display(multi_nagr_hypothesis_df)

    # Детализация на ключах для retl/term (как самых проблемных)
    details_metrics = ['retl_cnt', 'term_cnt']
    for dm in details_metrics:
        dm_pair = cmp_multi_max[['inn_key', 'agr_id_key', f'{dm}_lake', f'{dm}_excel', f'{dm}_abs']].merge(
            cmp_multi_sum[['inn_key', 'agr_id_key', f'{dm}_lake', f'{dm}_abs']],
            on=['inn_key', 'agr_id_key'],
            suffixes=('_max', '_sum')
        )
        dm_pair['improvement_sum_vs_max'] = dm_pair[f'{dm}_abs_max'] - dm_pair[f'{dm}_abs_sum']

        print(f"TOP ключей, где SUM лучше MAX для {dm}:")
        display(dm_pair.sort_values('improvement_sum_vs_max', ascending=False).head(30))

### 3.5 Кейс-диагностика `n_agr=30841`: почему `retl/term` не попадают в active

Секция детально проверяет воронку отбора терминалов по условиям active-логики:
- есть ли терминальный ID,
- не удалена ли запись,
- `d_ter_install <= month_end`,
- `d_ter_close >= month_start`.

Итог: показывает, на каком шаге происходит выпадение и почему `retl_cnt_lake`/`term_cnt_lake` становятся `NA` или сильно ниже Excel.

In [ ]:
# 3.5 Deep-dive по кейсу n_agr=30841
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров периода.')

case_n_agr = '30841'
case_month_start = pd.to_datetime(month_start)
case_month_end = pd.to_datetime(month_end)

# 1) Контекст договора
sql_case_context = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  cast(c.c_cmp_name as string) as company_name,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
from ods_alpha.scd1_agreements a
left join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where cast(a.n_agr as string) = '{case_n_agr}'
order by cast(a.d_valid_from as date) desc
"""

# 2) Все терминалы по цепочке n_agr -> n_cmp_client -> c_nmrc -> terminals (без active-фильтра)
sql_case_terminals = f"""
with agr as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.abs_agr_id as string) as agr_id
  from ods_alpha.scd1_agreements a
  where cast(a.n_agr as string) = '{case_n_agr}'
),
m_all as (
  select
    agr.agr_id,
    agr.n_agr,
    agr.n_cmp_client,
    cast(m.c_nmrc as string) as c_nmrc,
    coalesce(m.ods_deleted_flg, '0') as m_deleted_flg
  from agr
  join ods_alpha.scd1_merchants m
    on cast(m.n_cmp as string) = agr.n_cmp_client
  where m.c_nmrc is not null
  group by agr.agr_id, agr.n_agr, agr.n_cmp_client, cast(m.c_nmrc as string), coalesce(m.ods_deleted_flg, '0')
)
select
  m.agr_id,
  m.n_agr,
  m.n_cmp_client,
  m.c_nmrc,
  m.m_deleted_flg,
  cast(t.c_nter as string) as c_nter,
  cast(t.d_ter_install as date) as d_ter_install,
  cast(t.d_ter_close as date) as d_ter_close,
  coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
from m_all m
left join ods_alpha.scd1_pos_terminals t
  on cast(t.c_nmrc as string) = m.c_nmrc
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    case_context_df = imp.fetch(sql_case_context)
    case_terminals_df = imp.fetch(sql_case_terminals)

if case_context_df is None:
    case_context_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'company_name', 'inn'])
if case_terminals_df is None:
    case_terminals_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'c_nmrc', 'm_deleted_flg', 'c_nter', 'd_ter_install', 'd_ter_close', 't_deleted_flg'])

print(f'Кейс n_agr={case_n_agr}')
print(f'Контекст строк: {len(case_context_df):,}')
display(case_context_df)

if case_terminals_df.empty:
    print('По кейсу не найдено терминальных данных в цепочке merchants -> terminals.')
else:
    # Нормализация и флаги active-логики
    case_terminals_df['d_ter_install'] = pd.to_datetime(case_terminals_df['d_ter_install'], errors='coerce')
    case_terminals_df['d_ter_close'] = pd.to_datetime(case_terminals_df['d_ter_close'], errors='coerce')

    c_nter_series = case_terminals_df['c_nter'].astype(str).str.strip()
    case_terminals_df['is_c_nter_not_null'] = case_terminals_df['c_nter'].notna() & (~c_nter_series.isin(['', 'None', 'nan']))
    case_terminals_df['is_not_deleted'] = case_terminals_df['t_deleted_flg'].fillna('0').astype(str).ne('1')

    # Без fillna(Timestamp) — так стабильнее на текущей версии pandas
    case_terminals_df['is_install_ok'] = case_terminals_df['d_ter_install'].isna() | (case_terminals_df['d_ter_install'] <= case_month_end)
    case_terminals_df['is_close_ok'] = case_terminals_df['d_ter_close'].isna() | (case_terminals_df['d_ter_close'] >= case_month_start)

    case_terminals_df['is_active_term_logic'] = (
        case_terminals_df['is_c_nter_not_null']
        & case_terminals_df['is_not_deleted']
        & case_terminals_df['is_install_ok']
        & case_terminals_df['is_close_ok']
    )

    def _inactive_reason(row):
        if not row['is_c_nter_not_null']:
            return 'no_terminal_id'
        if not row['is_not_deleted']:
            return 'terminal_deleted'
        if not row['is_install_ok']:
            return 'install_after_month_end'
        if not row['is_close_ok']:
            return 'closed_before_month_start'
        return 'active_in_month'

    case_terminals_df['active_reason'] = case_terminals_df.apply(_inactive_reason, axis=1)

    # Сводка причин
    def _safe_term_nunique(series):
        s = series.astype(str).str.strip()
        s = s[~s.isin(['', 'None', 'nan', 'NaN', 'NaT'])]
        return int(s.nunique())

    reason_summary_df = (
        case_terminals_df.groupby('active_reason', as_index=False)
        .agg(
            rows=('c_nmrc', 'size'),
            retl_nunique=('c_nmrc', 'nunique'),
            term_nunique=('c_nter', _safe_term_nunique)
        )
        .sort_values('rows', ascending=False)
        .reset_index(drop=True)
    )

    # Воронка как в расчете active
    step_rows = [
        {
            'step': 'retl_all_cnt (без active-фильтра)',
            'value': int(case_terminals_df['c_nmrc'].nunique())
        },
        {
            'step': 'term_all_cnt (без active-фильтра)',
            'value': _safe_term_nunique(case_terminals_df.loc[case_terminals_df['is_c_nter_not_null'], 'c_nter'])
        },
        {
            'step': 'term_not_deleted_cnt',
            'value': _safe_term_nunique(case_terminals_df.loc[case_terminals_df['is_c_nter_not_null'] & case_terminals_df['is_not_deleted'], 'c_nter'])
        },
        {
            'step': 'term_install_ok_cnt',
            'value': _safe_term_nunique(case_terminals_df.loc[case_terminals_df['is_c_nter_not_null'] & case_terminals_df['is_not_deleted'] & case_terminals_df['is_install_ok'], 'c_nter'])
        },
        {
            'step': 'term_active_cnt (логика Lake)',
            'value': _safe_term_nunique(case_terminals_df.loc[case_terminals_df['is_active_term_logic'], 'c_nter'])
        },
        {
            'step': 'retl_active_cnt (логика Lake)',
            'value': int(case_terminals_df.loc[case_terminals_df['is_active_term_logic'], 'c_nmrc'].nunique())
        },
    ]
    case_funnel_df = pd.DataFrame(step_rows)

    print(f'Терминальных строк по кейсу: {len(case_terminals_df):,}')
    print('Сводка причин попадания/непопадания в active:')
    display(reason_summary_df)

    print('Воронка active-логики для кейса:')
    display(case_funnel_df)

    # Сравнение с cmp по agr_id кейса (если уже посчитано)
    if 'cmp' in globals() and isinstance(cmp, pd.DataFrame) and len(cmp):
        case_agr_id = None
        if len(case_context_df):
            case_agr_id = normalize_agr_q1(case_context_df['agr_id'].iloc[0])
        if case_agr_id:
            case_cmp_df = cmp[cmp['agr_id_key'] == case_agr_id].copy()
            print(f'Срез cmp для agr_id={case_agr_id}:')
            if not case_cmp_df.empty:
                display(case_cmp_df[['agr_id_key', 'inn_key', 'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta']])
            else:
                print('В cmp нет строки по этому agr_id (проверьте ключи/пересечение).')

    print('TOP-100 терминалов/точек с флагами active-логики:')
    display(
        case_terminals_df[[
            'agr_id', 'n_agr', 'n_cmp_client', 'c_nmrc', 'c_nter',
            'd_ter_install', 'd_ter_close', 't_deleted_flg',
            'is_c_nter_not_null', 'is_not_deleted', 'is_install_ok', 'is_close_ok',
            'is_active_term_logic', 'active_reason'
        ]]
        .sort_values(['is_active_term_logic', 'active_reason', 'c_nmrc', 'c_nter'], ascending=[True, True, True, True])
        .head(100)
    )

### 3.6 Deep-dive по `agr_id=1106410898346`: полная воронка `retl_cnt/term_cnt`

Секция выполняет end-to-end разбор по конкретному `agr_id`:
- все версии договора (`n_agr`) и контекст клиента;
- цепочка `agr_id -> n_agr -> n_cmp_client -> c_nmrc -> c_nter`;
- проверка active-условий для терминалов;
- воронка количеств по шагам;
- сверка с `cmp`/Excel для этого `agr_id`.

In [ ]:
# 3.6 AgrID funnel debug: 1106410898346
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров периода.')

case_agr_id = '1106410898346'
case_month_start = pd.to_datetime(month_start)
case_month_end = pd.to_datetime(month_end)

# 1) Контекст договоров по agr_id (все n_agr)
sql_agr_context = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  coalesce(a.ods_deleted_flg, '0') as agr_deleted_flg,
  cast(c.c_cmp_name as string) as company_name,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
from ods_alpha.scd1_agreements a
left join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where cast(a.abs_agr_id as string) = '{case_agr_id}'
order by cast(a.d_valid_from as date), cast(a.n_agr as string)
"""

# 2) Все точки/терминалы по цепочке (без active-фильтра)
sql_chain_all = f"""
with agr as (
  select distinct
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client
  from ods_alpha.scd1_agreements a
  where cast(a.abs_agr_id as string) = '{case_agr_id}'
),
m_all as (
  select
    agr.agr_id,
    agr.n_agr,
    agr.n_cmp_client,
    cast(m.c_nmrc as string) as c_nmrc,
    coalesce(m.ods_deleted_flg, '0') as m_deleted_flg
  from agr
  join ods_alpha.scd1_merchants m
    on cast(m.n_cmp as string) = agr.n_cmp_client
  where m.c_nmrc is not null
  group by agr.agr_id, agr.n_agr, agr.n_cmp_client, cast(m.c_nmrc as string), coalesce(m.ods_deleted_flg, '0')
)
select
  m.agr_id,
  m.n_agr,
  m.n_cmp_client,
  m.c_nmrc,
  m.m_deleted_flg,
  cast(t.c_nter as string) as c_nter,
  cast(t.d_ter_install as date) as d_ter_install,
  cast(t.d_ter_close as date) as d_ter_close,
  coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
from m_all m
left join ods_alpha.scd1_pos_terminals t
  on cast(t.c_nmrc as string) = m.c_nmrc
"""

# 3) Текущие lake-метрики из cmp/final_df для этого agr_id
case_cmp_df = pd.DataFrame()
if 'cmp' in globals() and isinstance(cmp, pd.DataFrame) and len(cmp):
    case_cmp_df = cmp[cmp['agr_id_key'] == case_agr_id].copy()

case_final_df = pd.DataFrame()
if 'final_df' in globals() and isinstance(final_df, pd.DataFrame) and len(final_df):
    ff = final_df.copy()
    ff['agr_id_key'] = ff['agr_id'].apply(normalize_agr_q1)
    case_final_df = ff[ff['agr_id_key'] == case_agr_id].copy()

with imp:
    imp.execute('set MEM_LIMIT=8g')
    agr_context_df = imp.fetch(sql_agr_context)
    chain_all_df = imp.fetch(sql_chain_all)

if agr_context_df is None:
    agr_context_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'agr_deleted_flg', 'company_name', 'inn'])
if chain_all_df is None:
    chain_all_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'c_nmrc', 'm_deleted_flg', 'c_nter', 'd_ter_install', 'd_ter_close', 't_deleted_flg'])

print(f'agr_id={case_agr_id}')
print(f'contracts rows: {len(agr_context_df):,}')
display(agr_context_df)

if chain_all_df.empty:
    print('Не найдены точки/терминалы по цепочке merchants -> terminals.')
else:
    chain_all_df['d_ter_install'] = pd.to_datetime(chain_all_df['d_ter_install'], errors='coerce')
    chain_all_df['d_ter_close'] = pd.to_datetime(chain_all_df['d_ter_close'], errors='coerce')

    c_nter_series = chain_all_df['c_nter'].astype(str).str.strip()
    chain_all_df['is_c_nter_not_null'] = chain_all_df['c_nter'].notna() & (~c_nter_series.isin(['', 'None', 'nan']))
    chain_all_df['is_not_deleted'] = chain_all_df['t_deleted_flg'].fillna('0').astype(str).ne('1')
    chain_all_df['is_install_ok'] = chain_all_df['d_ter_install'].isna() | (chain_all_df['d_ter_install'] <= case_month_end)
    chain_all_df['is_close_ok'] = chain_all_df['d_ter_close'].isna() | (chain_all_df['d_ter_close'] >= case_month_start)

    chain_all_df['is_active_term_logic'] = (
        chain_all_df['is_c_nter_not_null']
        & chain_all_df['is_not_deleted']
        & chain_all_df['is_install_ok']
        & chain_all_df['is_close_ok']
    )

    def _inactive_reason(row):
        if not row['is_c_nter_not_null']:
            return 'no_terminal_id'
        if not row['is_not_deleted']:
            return 'terminal_deleted'
        if not row['is_install_ok']:
            return 'install_after_month_end'
        if not row['is_close_ok']:
            return 'closed_before_month_start'
        return 'active_in_month'

    chain_all_df['active_reason'] = chain_all_df.apply(_inactive_reason, axis=1)

    def _safe_term_nunique(series):
        s = series.astype(str).str.strip()
        s = s[~s.isin(['', 'None', 'nan', 'NaN', 'NaT'])]
        return int(s.nunique())

    # 4) Воронка по каждому n_agr
    funnel_rows = []
    for n_agr_val, g in chain_all_df.groupby('n_agr'):
        funnel_rows.extend([
            {'n_agr': n_agr_val, 'step': 'retl_all_cnt', 'value': int(g['c_nmrc'].nunique())},
            {'n_agr': n_agr_val, 'step': 'term_all_cnt', 'value': _safe_term_nunique(g.loc[g['is_c_nter_not_null'], 'c_nter'])},
            {'n_agr': n_agr_val, 'step': 'term_not_deleted_cnt', 'value': _safe_term_nunique(g.loc[g['is_c_nter_not_null'] & g['is_not_deleted'], 'c_nter'])},
            {'n_agr': n_agr_val, 'step': 'term_install_ok_cnt', 'value': _safe_term_nunique(g.loc[g['is_c_nter_not_null'] & g['is_not_deleted'] & g['is_install_ok'], 'c_nter'])},
            {'n_agr': n_agr_val, 'step': 'term_active_cnt', 'value': _safe_term_nunique(g.loc[g['is_active_term_logic'], 'c_nter'])},
            {'n_agr': n_agr_val, 'step': 'retl_active_cnt', 'value': int(g.loc[g['is_active_term_logic'], 'c_nmrc'].nunique())},
        ])

    agr_funnel_df = pd.DataFrame(funnel_rows)

    # 5) Сводка причин выпадения по n_agr
    reason_summary_df = (
        chain_all_df.groupby(['n_agr', 'active_reason'], as_index=False)
        .agg(
            rows=('c_nmrc', 'size'),
            retl_nunique=('c_nmrc', 'nunique'),
            term_nunique=('c_nter', _safe_term_nunique)
        )
        .sort_values(['n_agr', 'rows'], ascending=[True, False])
        .reset_index(drop=True)
    )

    print(f'chain rows: {len(chain_all_df):,}; n_agr involved: {chain_all_df["n_agr"].nunique():,}')
    print('Воронка retl/term по каждому n_agr:')
    display(agr_funnel_df)

    print('Причины выпадения из active по каждому n_agr:')
    display(reason_summary_df)

    print('Сырые строки chain с флагами (TOP-200):')
    display(
        chain_all_df[[
            'agr_id', 'n_agr', 'n_cmp_client', 'c_nmrc', 'm_deleted_flg', 'c_nter',
            'd_ter_install', 'd_ter_close', 't_deleted_flg',
            'is_c_nter_not_null', 'is_not_deleted', 'is_install_ok', 'is_close_ok',
            'is_active_term_logic', 'active_reason'
        ]]
        .sort_values(['n_agr', 'is_active_term_logic', 'active_reason', 'c_nmrc', 'c_nter'], ascending=[True, True, True, True, True])
        .head(200)
    )

# 6) Сверка с cmp/final_df/Excel
print('\nСрез final_df по agr_id:')
if case_final_df.empty:
    print('В final_df нет строк по этому agr_id.')
else:
    display(case_final_df[['report_month', 'inn', 'agr_id', 'n_agr', 'retl_cnt', 'term_cnt', 'active_term_cnt', 'company_name']])

print('Срез cmp по agr_id:')
if case_cmp_df.empty:
    print('В cmp нет строки по этому agr_id (проверьте пересечение ключей).')
else:
    display(case_cmp_df[['agr_id_key', 'inn_key', 'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta']])

### 3.7 Разбор кейса `agr_id=1106410898346` через `n_agr=45011`

Секция проверяет путь формирования метрик именно через `n_agr=45011` и отвечает на вопрос:
- находится ли `agr_id=1106410898346` в SA-периметре как `abs_agr_id`;
- находится ли `n_agr=45011` в SA-периметре;
- не подставился ли `agr_id` через fallback-логику.

In [ ]:
# 3.7 Path debug: agr_id=1106410898346 via n_agr=45011
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров периода.')

case_agr_id = '1106410898346'
case_n_agr = '45011'

# 1) Проверка в уже посчитанном sa_df (если есть)
sa_df_check_df = pd.DataFrame()
if 'sa_df' in globals() and isinstance(sa_df, pd.DataFrame) and len(sa_df):
    tmp = sa_df.copy()
    tmp['agr_id_key'] = tmp['agr_id'].apply(normalize_agr_q1)
    tmp['n_agr_key'] = tmp['n_agr'].astype(str)
    sa_df_check_df = tmp[(tmp['agr_id_key'] == case_agr_id) | (tmp['n_agr_key'] == case_n_agr)].copy()

# 2) Проверка SA-периметра напрямую SQL (по тем же правилам, что в секции 01)
sql_sa_case_check = f"""
select distinct
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name,
  coalesce(a.ods_deleted_flg, '0') as agr_deleted_flg,
  upper(trim(cast(a.acq_class as string))) as acq_class
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
  and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
  )
  and (
      cast(a.abs_agr_id as string) = '{case_agr_id}'
      or cast(a.n_agr as string) = '{case_n_agr}'
  )
order by cast(a.d_valid_from as date), cast(a.n_agr as string)
"""

# 3) Проверка agreements без SA/terms-фильтра, чтобы увидеть связь n_agr -> abs_agr_id
sql_agreements_raw_check = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  coalesce(a.ods_deleted_flg, '0') as agr_deleted_flg,
  upper(trim(cast(a.acq_class as string))) as acq_class
from ods_alpha.scd1_agreements a
where cast(a.abs_agr_id as string) = '{case_agr_id}'
   or cast(a.n_agr as string) = '{case_n_agr}'
order by cast(a.d_valid_from as date), cast(a.n_agr as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    sa_sql_check_df = imp.fetch(sql_sa_case_check)
    agreements_raw_check_df = imp.fetch(sql_agreements_raw_check)

if sa_sql_check_df is None:
    sa_sql_check_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'inn', 'company_name', 'agr_deleted_flg', 'acq_class'])
if agreements_raw_check_df is None:
    agreements_raw_check_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'agr_deleted_flg', 'acq_class'])

# 4) Срез base_df/final_df для понимания пути подстановки
base_case_df = pd.DataFrame()
if 'base_df' in globals() and isinstance(base_df, pd.DataFrame) and len(base_df):
    b = base_df.copy()
    b['agr_id_key'] = b['agr_id'].apply(normalize_agr_q1)
    b['n_agr_key'] = b['n_agr'].astype(str)
    keep_cols = [
        'agr_id', 'n_agr', 'agr_id_source', 'r2_id', 'n_cmp_client', 'cft_id',
        'company_name', 'inn', 'retl_cnt', 'term_cnt', 'active_term_cnt'
    ]
    keep_cols = [c for c in keep_cols if c in b.columns]
    base_case_df = b[(b['agr_id_key'] == case_agr_id) | (b['n_agr_key'] == case_n_agr)][keep_cols].copy()

final_case_df = pd.DataFrame()
if 'final_df' in globals() and isinstance(final_df, pd.DataFrame) and len(final_df):
    f = final_df.copy()
    f['agr_id_key'] = f['agr_id'].apply(normalize_agr_q1)
    f['n_agr_key'] = f['n_agr'].astype(str)
    final_case_df = f[(f['agr_id_key'] == case_agr_id) | (f['n_agr_key'] == case_n_agr)].copy()

# 5) Явный ответ на ключевой вопрос
sa_sql_check_df['agr_id_key'] = sa_sql_check_df['agr_id'].apply(normalize_agr_q1) if len(sa_sql_check_df) else pd.Series(dtype=object)
sa_sql_check_df['n_agr_key'] = sa_sql_check_df['n_agr'].astype(str) if len(sa_sql_check_df) else pd.Series(dtype=object)

is_agr_id_in_sa = bool((sa_sql_check_df['agr_id_key'] == case_agr_id).any()) if len(sa_sql_check_df) else False
is_n_agr_in_sa = bool((sa_sql_check_df['n_agr_key'] == case_n_agr).any()) if len(sa_sql_check_df) else False

print(f'Проверка кейса: agr_id={case_agr_id}, n_agr={case_n_agr}')
print(f'agr_id в SA-периметре (как abs_agr_id): {is_agr_id_in_sa}')
print(f'n_agr в SA-периметре: {is_n_agr_in_sa}')

if is_n_agr_in_sa and not is_agr_id_in_sa:
    print('Вывод: метрики идут через n_agr, а agr_id в итоге, вероятно, подставляется fallback-логикой (не как прямой abs_agr_id из SA).')

print('\nSA check via sa_df (если sa_df в памяти):')
if sa_df_check_df.empty:
    print('Нет строк в sa_df по этому agr_id/n_agr (или sa_df не в памяти).')
else:
    display(sa_df_check_df[['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'company_name', 'inn']])

print('\nSA check via SQL perimeter rules:')
display(sa_sql_check_df)

print('\nRaw agreements check (без SA/terms-фильтра):')
display(agreements_raw_check_df)

print('\nbase_df path check:')
if base_case_df.empty:
    print('В base_df нет строк по этому agr_id/n_agr.')
else:
    display(base_case_df)

print('\nfinal_df path check:')
if final_case_df.empty:
    print('В final_df нет строк по этому agr_id/n_agr.')
else:
    display(final_case_df[['report_month', 'inn', 'agr_id', 'n_agr', 'retl_cnt', 'term_cnt', 'active_term_cnt', 'company_name']])

if 'cmp' in globals() and isinstance(cmp, pd.DataFrame) and len(cmp):
    cmp_case_df = cmp[cmp['agr_id_key'] == case_agr_id].copy()
    print('\ncmp check by agr_id:')
    if cmp_case_df.empty:
        print('В cmp нет строк по этому agr_id.')
    else:
        display(cmp_case_df[['agr_id_key', 'inn_key', 'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta']])

### 3.8 Проверка R2-слоя: где может лежать `retl=-307` для fallback `agr_id`

Идея проверки:
- берем `agr_id=1106410898346` (fallback из `r2_id`) и его `cft_id`;
- в `ods.scd1_z_r2_ip_merchants` собираем **все** `r2_id` этого `cft_id`, считаем `retl` (`c_nmrc`) и терминалы;
- сравниваем `retl` из R2 с `retl` из `cmp` (`Lake`/`Excel`) и смотрим, может ли "потерянный" объем сидеть на других `r2_id` того же клиента.

In [ ]:
# 3.8 R2-layer check: where retl=-307 may hide for fallback agr_id
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров месяца.')

case_agr_id = '1106410898346'
case_n_agr = '45011'

# 1) Контекст из base_df/cmp
base_case_38_df = pd.DataFrame()
if 'base_df' in globals() and isinstance(base_df, pd.DataFrame) and len(base_df):
    b = base_df.copy()
    b['agr_id_key'] = b['agr_id'].apply(normalize_agr_q1)
    b['n_agr_key'] = b['n_agr'].astype(str)
    keep_cols = [
        'agr_id', 'n_agr', 'agr_id_source', 'r2_id', 'cft_id',
        'n_cmp_client', 'company_name', 'inn', 'retl_cnt', 'term_cnt', 'active_term_cnt'
    ]
    keep_cols = [c for c in keep_cols if c in b.columns]
    base_case_38_df = b[(b['agr_id_key'] == case_agr_id) | (b['n_agr_key'] == case_n_agr)][keep_cols].copy()

cmp_case_38_df = pd.DataFrame()
if 'cmp' in globals() and isinstance(cmp, pd.DataFrame) and len(cmp):
    cmp_case_38_df = cmp[cmp['agr_id_key'] == case_agr_id].copy()

# 2) Определяем cft_id (сначала из base_df, если нет — из R2 по id)
cft_candidates = []
if not base_case_38_df.empty and 'cft_id' in base_case_38_df.columns:
    cft_candidates = sorted({
        str(x).strip()
        for x in base_case_38_df['cft_id'].dropna().tolist()
        if str(x).strip() not in ['', 'None', 'nan']
    })

cft_id_for_check = cft_candidates[0] if cft_candidates else None
cft_from_r2_df = pd.DataFrame(columns=['cft_id'])

if cft_id_for_check is None:
    sql_cft_from_r2 = f"""
    select distinct cast(m.c_cl_org as string) as cft_id
    from ods.scd1_z_r2_ip_merchants m
    where cast(m.id as string) = '{case_agr_id}'
    """
    with imp:
        imp.execute('set MEM_LIMIT=4g')
        cft_from_r2_df = imp.fetch(sql_cft_from_r2)

    if cft_from_r2_df is None:
        cft_from_r2_df = pd.DataFrame(columns=['cft_id'])

    if len(cft_from_r2_df):
        vals = [str(x).strip() for x in cft_from_r2_df['cft_id'].dropna().tolist() if str(x).strip()]
        if vals:
            cft_id_for_check = vals[0]

print(f'Кейс 3.8: agr_id={case_agr_id}, n_agr={case_n_agr}')
print(f'cft_id для проверки: {cft_id_for_check}')

if not base_case_38_df.empty:
    print('\nКонтекст из base_df:')
    display(base_case_38_df)

if not cft_from_r2_df.empty:
    print('\nCFT из R2 по fallback agr_id:')
    display(cft_from_r2_df)

if cft_id_for_check is None:
    print('Не удалось определить cft_id. Выполните секции 01-10 и 3.7, затем повторите 3.8.')
else:
    cft_sql = cft_id_for_check.replace("'", "''")

    # 3) Проверка схемы R2-таблицы, чтобы не ссылаться на несуществующие поля
    with imp:
        imp.execute('set MEM_LIMIT=2g')
        r2_desc_df = imp.fetch('describe ods.scd1_z_r2_ip_merchants')

    if r2_desc_df is None:
        r2_desc_df = pd.DataFrame()

    if r2_desc_df.empty:
        print('DESCRIBE ods.scd1_z_r2_ip_merchants вернул пустой результат.')
    else:
        name_col = None
        for cand in ['name', 'col_name', 'column_name']:
            if cand in r2_desc_df.columns:
                name_col = cand
                break
        if name_col is None:
            name_col = r2_desc_df.columns[0]

        r2_cols = {str(x).strip().lower() for x in r2_desc_df[name_col].dropna().tolist()}
        required_cols = {'id', 'c_cl_org', 'c_nmrc'}
        missing_cols = sorted(required_cols - r2_cols)

        if missing_cols:
            print(f'В ods.scd1_z_r2_ip_merchants не найдены колонки: {missing_cols}')
            print('Проверьте схему таблицы ниже.')
            display(r2_desc_df.head(200))
        else:
            # 4) Все merchant-строки по одному cft_id
            sql_r2_retl_rows = f"""
            select
              cast(m.id as string) as r2_id,
              cast(m.c_cl_org as string) as cft_id,
              cast(m.c_nmrc as string) as c_nmrc,
              coalesce(m.ods_deleted_flg, '0') as m_deleted_flg,
              cast(m.c_tariff_plan as string) as c_tariff_plan
            from ods.scd1_z_r2_ip_merchants m
            where cast(m.c_cl_org as string) = '{cft_sql}'
              and m.c_nmrc is not null
            """

            # 5) Агрегаты по каждому r2_id клиента
            sql_r2_counts_by_id = f"""
            with m as (
              select
                cast(id as string) as r2_id,
                cast(c_nmrc as string) as c_nmrc
              from ods.scd1_z_r2_ip_merchants
              where cast(c_cl_org as string) = '{cft_sql}'
                and c_nmrc is not null
            ),
            mt as (
              select
                m.r2_id,
                m.c_nmrc,
                cast(t.c_nter as string) as c_nter,
                cast(t.d_ter_install as date) as d_ter_install,
                cast(t.d_ter_close as date) as d_ter_close,
                coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
              from m
              left join ods_alpha.scd1_pos_terminals t
                on cast(t.c_nmrc as string) = m.c_nmrc
            )
            select
              r2_id,
              count(distinct c_nmrc) as retl_all_cnt,
              count(distinct case when c_nter is not null then c_nter else null end) as term_all_cnt,
              count(distinct case
                when c_nter is not null
                 and t_deleted_flg <> '1'
                 and (d_ter_install is null or d_ter_install <= cast('{month_end}' as date))
                 and (d_ter_close is null or d_ter_close >= cast('{month_start}' as date))
                then c_nmrc else null end
              ) as retl_active_cnt,
              count(distinct case
                when c_nter is not null
                 and t_deleted_flg <> '1'
                 and (d_ter_install is null or d_ter_install <= cast('{month_end}' as date))
                 and (d_ter_close is null or d_ter_close >= cast('{month_start}' as date))
                then c_nter else null end
              ) as term_active_cnt
            from mt
            group by r2_id
            order by retl_all_cnt desc, r2_id
            """

            # 6) Агрегаты по клиенту целиком (по всем его r2_id)
            sql_r2_counts_total = f"""
            with m as (
              select
                cast(id as string) as r2_id,
                cast(c_nmrc as string) as c_nmrc
              from ods.scd1_z_r2_ip_merchants
              where cast(c_cl_org as string) = '{cft_sql}'
                and c_nmrc is not null
            ),
            mt as (
              select
                m.r2_id,
                m.c_nmrc,
                cast(t.c_nter as string) as c_nter,
                cast(t.d_ter_install as date) as d_ter_install,
                cast(t.d_ter_close as date) as d_ter_close,
                coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
              from m
              left join ods_alpha.scd1_pos_terminals t
                on cast(t.c_nmrc as string) = m.c_nmrc
            )
            select
              count(distinct c_nmrc) as retl_all_cnt,
              count(distinct case when c_nter is not null then c_nter else null end) as term_all_cnt,
              count(distinct case
                when c_nter is not null
                 and t_deleted_flg <> '1'
                 and (d_ter_install is null or d_ter_install <= cast('{month_end}' as date))
                 and (d_ter_close is null or d_ter_close >= cast('{month_start}' as date))
                then c_nmrc else null end
              ) as retl_active_cnt,
              count(distinct case
                when c_nter is not null
                 and t_deleted_flg <> '1'
                 and (d_ter_install is null or d_ter_install <= cast('{month_end}' as date))
                 and (d_ter_close is null or d_ter_close >= cast('{month_start}' as date))
                then c_nter else null end
              ) as term_active_cnt
            from mt
            """

            with imp:
                imp.execute('set MEM_LIMIT=8g')
                r2_retl_rows_df = imp.fetch(sql_r2_retl_rows)
                r2_counts_by_id_df = imp.fetch(sql_r2_counts_by_id)
                r2_counts_total_df = imp.fetch(sql_r2_counts_total)

            if r2_retl_rows_df is None:
                r2_retl_rows_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'c_nmrc', 'm_deleted_flg', 'c_tariff_plan'])
            if r2_counts_by_id_df is None:
                r2_counts_by_id_df = pd.DataFrame(columns=['r2_id', 'retl_all_cnt', 'term_all_cnt', 'retl_active_cnt', 'term_active_cnt'])
            if r2_counts_total_df is None:
                r2_counts_total_df = pd.DataFrame(columns=['retl_all_cnt', 'term_all_cnt', 'retl_active_cnt', 'term_active_cnt'])

            if not r2_retl_rows_df.empty:
                r2_retl_rows_df['r2_id'] = r2_retl_rows_df['r2_id'].astype(str)
                r2_retl_rows_df['c_nmrc'] = r2_retl_rows_df['c_nmrc'].astype(str).str.strip()

            for col in ['retl_all_cnt', 'term_all_cnt', 'retl_active_cnt', 'term_active_cnt']:
                if col in r2_counts_by_id_df.columns:
                    r2_counts_by_id_df[col] = pd.to_numeric(r2_counts_by_id_df[col], errors='coerce').fillna(0).astype(int)
                if col in r2_counts_total_df.columns:
                    r2_counts_total_df[col] = pd.to_numeric(r2_counts_total_df[col], errors='coerce').fillna(0).astype(int)

            def _max_num(df, col):
                if df is None or not isinstance(df, pd.DataFrame) or df.empty or col not in df.columns:
                    return None
                s = pd.to_numeric(df[col], errors='coerce').dropna()
                if s.empty:
                    return None
                return float(s.max())

            retl_excel = _max_num(cmp_case_38_df, 'retl_cnt_excel')
            retl_lake = _max_num(cmp_case_38_df, 'retl_cnt_lake')

            retl_focus_id = 0
            if not r2_counts_by_id_df.empty:
                m_focus = r2_counts_by_id_df['r2_id'].astype(str) == case_agr_id
                if m_focus.any():
                    retl_focus_id = int(r2_counts_by_id_df.loc[m_focus, 'retl_all_cnt'].max())

            retl_total_r2 = 0
            retl_active_total_r2 = 0
            if not r2_counts_total_df.empty:
                retl_total_r2 = int(r2_counts_total_df['retl_all_cnt'].iloc[0])
                retl_active_total_r2 = int(r2_counts_total_df['retl_active_cnt'].iloc[0])

            retl_other_ids = max(retl_total_r2 - retl_focus_id, 0)

            summary_rows = [
                {'metric': 'retl_cnt_excel (cmp)', 'value': retl_excel},
                {'metric': 'retl_cnt_lake (cmp)', 'value': retl_lake},
                {'metric': 'retl_cnt_r2_focus_id_all', 'value': retl_focus_id},
                {'metric': 'retl_cnt_r2_other_ids_all', 'value': retl_other_ids},
                {'metric': 'retl_cnt_r2_all_ids_all', 'value': retl_total_r2},
                {'metric': 'retl_cnt_r2_all_ids_active_logic', 'value': retl_active_total_r2},
            ]
            if retl_excel is not None and retl_lake is not None:
                summary_rows.append({'metric': 'excel_minus_lake', 'value': retl_excel - retl_lake})
            if retl_excel is not None:
                summary_rows.append({'metric': 'excel_minus_r2_all_ids', 'value': retl_excel - retl_total_r2})
                summary_rows.append({'metric': 'excel_minus_r2_active_logic', 'value': retl_excel - retl_active_total_r2})
            if retl_lake is not None:
                summary_rows.append({'metric': 'r2_all_ids_minus_lake', 'value': retl_total_r2 - retl_lake})

            r2_38_summary_df = pd.DataFrame(summary_rows)

            print('\nR2 coverage summary for same cft_id:')
            display(r2_38_summary_df)

            print('\nR2 counts by r2_id (same cft_id):')
            display(r2_counts_by_id_df)

            print('\nRows in R2 merchants for same cft_id (TOP-200):')
            display(r2_retl_rows_df.sort_values(['r2_id', 'c_nmrc']).head(200))

            if not cmp_case_38_df.empty:
                cmp_cols = [
                    'agr_id_key', 'inn_key',
                    'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta',
                    'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta'
                ]
                cmp_cols = [c for c in cmp_cols if c in cmp_case_38_df.columns]
                print('\ncmp slice for case agr_id:')
                display(cmp_case_38_df[cmp_cols])